# エージェントとしてのNVIDIA NIM

このノートブックは、LangGraphを用いて固定されたワークフローに沿ってツールを呼び出し、実行するエージェントについて説明します。マルチエージェントを含む自律的なエージェントを効率的に開発するツールキットとしてはNeMo Agent Toolkitがあります。NeMo Agent Toolkitについてはこのノートブックでは触れませんが、興味がある方は[こちら](https://developer.nvidia.com/nemo-agent-toolkit)をご覧ください

ツールとは、入力を受け取り、アクションを実行し、定義済みスキーマに従った構造化出力で結果を返すインターフェースです。多くの場合、外部API呼び出しを含み、LLM単体ではできないタスクを実行できます。ただし、必ずしも外部API呼び出しである必要はありません。例えば、サンディエゴの天気を取得するには天気ツール、49ersの試合結果を取得するにはウェブ検索ツールやESPNツールなどが使えます。
AIエージェントの詳細は [こちら](https://www.nvidia.com/en-us/glossary/ai-agents/)をご覧ください。

<div style="text-align: center;">
    <img src="imgs/agent-components.png" alt="Agent Overall Workflow" style="width: 80%; max-width: 700px;"> <br>
</div>
<br/>

このノートブックでは、以下のエージェント的な構成でNIMを利用します:
### LLM
* ローカルでホストされたLlama 3.1 8Bモデルのエンドポイント

### ツール
* 画像キャプション用のローカルNIM（Llama 3.2 vision instructモデル使用）
* リアルタイム情報取得用APIベースのツール（Tavily Search）

## 環境設定

In [ ]:
import getpass
import os

# del os.environ['NVIDIA_API_KEY']  ## delete key and reset
if os.environ.get("NVIDIA_API_KEY", "").startswith("nvapi-"):
    print("Valid NVIDIA_API_KEY already in environment. Delete to reset")
else:
    nvapi_key = getpass.getpass("NVAPI Key (starts with nvapi-): ")
    assert nvapi_key.startswith("nvapi-"), f"{nvapi_key[:5]}... is not a valid key"
    os.environ["NVIDIA_API_KEY"] = nvapi_key
    os.environ["NGC_API_KEY"] = nvapi_key

global nvapi_key

### ローカルVLMのデプロイ

ローカルにデプロイされたVLM NIMは、エージェントが呼び出す最初のツールとなります。VLM NIMは画像にキャプションを付けるために使用されます。

利用可能なポートを検索

In [ ]:
import random
import socket
import os 

def find_available_port(start=11000, end=11999):
    while True:
        # Randomly select a port between start and end range
        port = random.randint(start, end)
        
        # Try to create a socket and bind to the port
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
            try:
                sock.bind(("localhost", port))
                # If binding is successful, the port is free
                return port
            except OSError:
                # If binding fails, the port is in use, continue to the next iteration
                continue

# Find and print an available port
os.environ['CONTAINER_PORT'] = str(find_available_port())
print(f"Your have been alloted the available port: {os.environ['CONTAINER_PORT']}")

In [ ]:
from os.path import expanduser
home = expanduser("~")
os.environ['LOCAL_NIM_CACHE']=f"{home}/.cache/nim"
!echo $LOCAL_NIM_CACHE

In [ ]:
!mkdir -p "$LOCAL_NIM_CACHE"
!chmod 777 "$LOCAL_NIM_CACHE"

In [ ]:
! echo -e "$NGC_API_KEY" | docker login nvcr.io --username '$oauthtoken' --password-stdin

In [ ]:
! docker run -it --rm -d \
--gpus '"device=0"' \
--name=vlm_nim \
--shm-size=16GB  \
-e NGC_API_KEY \
-v $LOCAL_NIM_CACHE:/opt/nim/.cache \
-u $(id -u) \
-p $CONTAINER_PORT:8000 \
nvcr.io/nim/meta/llama-3.2-11b-vision-instruct:1.1.1

# -e NIM_MODEL_PROFILE=tensorrt_llm-a100-bf16-tp1-throughput \
# In order to ensure, the local NIM container is completely loaded and doesn't remain in pending stage, we instantiate a wait interval
! sleep 60

In [ ]:
! docker ps -a | awk 'NR==1 || /llama/'

### ローカルLLMのデプロイ



今回使用するLLMは `tokyotech-llm/llama-3.1-swallow-8b-instruct-v0.1` です。

モデルは1台のA100/H100 80GB GPUで実行できます。

In [ ]:
import random
import socket
import os 

def find_available_port(start=11000, end=11999):
    while True:
        # Randomly select a port between start and end range
        port = random.randint(start, end)
        
        # Try to create a socket and bind to the port
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
            try:
                sock.bind(("localhost", port))
                # If binding is successful, the port is free
                return port
            except OSError:
                # If binding fails, the port is in use, continue to the next iteration
                continue

# Find and print an available port
os.environ['LLM_CONTAINER_PORT'] = str(find_available_port())
print(f"Your have been alloted the available port: {os.environ['LLM_CONTAINER_PORT']}")

In [ ]:
!export LOCAL_NIM_CACHE=~/.cache/nim
!mkdir -p "$LOCAL_NIM_CACHE"

!docker run -it -d --rm \
    --gpus '"device=1"' \
    --name=llm_nim \
    --shm-size=16GB \
    -e NGC_API_KEY \
    -v "$LOCAL_NIM_CACHE:/opt/nim/.cache" \
    -u $(id -u) \
    -p $LLM_CONTAINER_PORT:8000 \
    nvcr.io/nim/tokyotech-llm/llama-3.1-swallow-8b-instruct-v0.1:latest

!sleep 60

## ツール呼び出しの設定

In [ ]:
# === Imports ===
import base64, io
from PIL import Image
from typing import Annotated, TypedDict
from pydantic import BaseModel, Field

from langchain_core.tools import tool
from langchain_core.runnables import RunnableLambda
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langgraph.graph import END, StateGraph
from langchain_community.tools.tavily_search import TavilySearchResults

# === Image-to-base64 ===
def img2base64_string(img_path):
    print(img_path)
    image = Image.open(img_path)
    if image.width > 800 or image.height > 800:
        image.thumbnail((800, 800))
    buffered = io.BytesIO()
    image.convert("RGB").save(buffered, format="JPEG", quality=85)
    return base64.b64encode(buffered.getvalue()).decode()

### === ツール: 画像キャプション生成 ===

In [ ]:
@tool
def ImageCaptionTool(img_path: Annotated[str, "入力画像のパス"]) -> str:
    """vision LLMを使って画像の状況を記述して.英語で出力する"""
    image_b64 = img2base64_string(img_path)
    prompt = f'What is this image？ <img src="data:image/jpeg;base64,{image_b64}" />'
    
    chat_model = ChatNVIDIA(
        base_url=f"http://0.0.0.0:{os.environ['CONTAINER_PORT']}/v1",
        model="meta/llama-3.2-11b-vision-instruct",
        max_tokens=512,
        temperature=0.00,
        top_p=1.00
    )
    
    response = chat_model.invoke(prompt)
    return response.content

画像キャプションツールが正しく実行されているか確認します。

In [ ]:
img_path="./imgs/place_pic.jpg"

out=ImageCaptionTool(img_path)
out

### 期待される出力

```
./imgs/place_pic.jpg
'This image depicts the Taj Mahal, a mausoleum located in Agra, India, built by Mughal Emperor Shah Jahan in memory of his wife, Mumtaz Mahal, who died during the birth of their 14th child. The Taj Mahal is an example of Mughal architecture, which combines elements of Indian, Persian, and Islamic styles. It is considered one of the most beautiful buildings in the world and is a UNESCO World Heritage Site.\n\nThe Taj Mahal is made of white marble and features intricate carvings, calligraphy, and inlays of precious stones such as jasper, jade, and turquoise. The main structure is surrounded by gardens and a reflecting pool, which creates a sense of symmetry and balance. The Taj Mahal is also decorated with flowers, birds, and other motifs, reflecting the Islamic tradition of using geometric patterns and natural forms in architecture.
```

In [ ]:
img_path="imgs/place_pic1.jpg"

out=ImageCaptionTool(img_path)
out

In [ ]:
# === Location extractors ===
class ImgPathExtractor(BaseModel):
    img_path: str = Field(description="入力画像へのパス")

class LocationExtractor(BaseModel):
    location: str = Field(description="記述もしくは言及された場所の名前")

# Using the 405B model for excellent tool calling capabilities
#llm = ChatNVIDIA(model="meta/llama-3.1-405b-instruct", max_tokens=1024)

# OR USE the locally deployed variant.
llm = ChatNVIDIA(model="tokyotech-llm/llama-3.1-swallow-8b-instruct-v0.1", \
                 max_tokens=1024,\
                 base_url="http://0.0.0.0:{}/v1".format(os.environ['LLM_CONTAINER_PORT'])
                )


# === Parser functions ===
format_chain = (
    ChatPromptTemplate.from_template("inputから画像パスを抽出: {input}")
    | llm.with_structured_output(ImgPathExtractor)
)

def parse_image_path(state) -> dict:
    structured = format_chain.invoke({"input": state["input"]})
    return {"img_path": structured.img_path}

def extract_caption(state) -> dict:
    result = ImageCaptionTool.invoke({"img_path": state["img_path"]})
    return {"image_caption": result}

caption_to_location = (
    ChatPromptTemplate.from_template("caption中の画像の記述から場所を抽出: {caption}")
    | llm.with_structured_output(LocationExtractor)
)

In [ ]:
def extract_location_from_caption(state) -> dict:
    print(state)
    result = caption_to_location.invoke({"caption": state["image_caption"]})
    return {"location": result.location}

text_location_chain = (
    ChatPromptTemplate.from_template("このユーザークエリから場所を抽出: {input}")
    | llm.with_structured_output(LocationExtractor)
)

def extract_location_from_text(state) -> dict:
    result = text_location_chain.invoke({"input": state["input"]})
    # Add default values for image-related fields so summarizer doesn't fail
    return {
        "location": result.location,
        "img_path": "N/A",  # Default value
        "image_caption": "No image provided"  # Default value
    }

### === ツール: Web検索 ===

[Tavily](https://docs.tavily.com/documentation/about)は、効率的で迅速かつ永続的な検索結果を目指したLLM向けに最適化された検索エンジンです。SerpやGoogleなどの他の検索APIとは異なり、TavilyはAI開発者と自律型AIエージェント向けの検索の最適化に重点を置いています。オンラインソースからの検索、スクレイピング、フィルタリング、最も関連性の高い情報の抽出など、すべての負担を1回のAPI呼び出しで処理します！

In [ ]:
from langchain_community.tools.tavily_search import TavilySearchResults

In [ ]:
import getpass
import os

if "TAVILY_API_KEY" not in os.environ:
    os.environ["TAVILY_API_KEY"] = getpass.getpass("Enter your Tavily API key")

In [ ]:
weather_search_tool = TavilySearchResults(max_results=2)

def search_weather(state) -> dict:
    query = f"current weather in {state['location']}"
    result = weather_search_tool.invoke(query)
    return {"search_results": result}

In [ ]:
summary_prompt = ChatPromptTemplate.from_template(
    """あなたは、視覚情報とウェブデータをもとに要約するアシスタントです。

ウェブ検索結果: {search_results}

これらの情報を使って、その場所の現在の天気を2～3行でわかりやすく要約してください。また、その天気に合ったおすすめのアクティビティも提案してください。
"""
)

summarizer_chain = summary_prompt | llm | StrOutputParser()

def summarize_result(state) -> dict:
    summary = summarizer_chain.invoke(state)
    return {"final_response": summary}

# === Routing node ===
def route_input(state) -> dict:
    if any(ext in state["input"] for ext in [".png", ".jpg", ".jpeg"]):
        return {"has_image": True}
    else:
        return {"has_image": False}

## LangGraph セットアップ

[LangGraph](https://langchain-ai.github.io/langgraph/)は、制御可能なエージェントを構築するための低レベルのオーケストレーションフレームワークです。Langchainが LLMアプリケーション開発を効率化するための統合機能とコンポーネントを提供する一方、LangGraphライブラリはエージェントのオーケストレーションを可能にし、カスタマイズ可能なアーキテクチャ、長期メモリ、および人間参加型のワークフローを提供して、複雑なタスクを確実に処理します。

### ステートのセットアップ

グラフを定義する際の最初のステップは、グラフのステートを定義することです。
ステートは、グラフのスキーマと、ステートの更新方法を指定するリデューサー関数で構成されています。ステートのスキーマは、グラフ内のすべてのノードとエッジへの入力スキーマとなり、TypedDictまたはPydanticモデルのいずれかを使用できます。すべてのノードはステートへの更新を発行し、それらは指定されたリデューサー関数を使用して適用されます。

In [ ]:
# === State type ===
class ImageWeatherState(TypedDict):
    input: str
    img_path: str
    image_caption: str
    location: str
    search_results: str
    final_response: str
    has_image: bool

# === Build LangGraph ===
builder = StateGraph(ImageWeatherState)

builder.add_node("route_input", RunnableLambda(route_input))
builder.add_node("parse_image_path", RunnableLambda(parse_image_path))
builder.add_node("caption_image", RunnableLambda(extract_caption))
builder.add_node("extract_location_from_caption", RunnableLambda(extract_location_from_caption))
builder.add_node("extract_location_from_text", RunnableLambda(extract_location_from_text))
builder.add_node("search_weather", RunnableLambda(search_weather))
builder.add_node("summarize", RunnableLambda(summarize_result))

builder.set_entry_point("route_input")

builder.add_conditional_edges(
    source="route_input",
    path=lambda state: "has_image" if state["has_image"] else "text_only",
    path_map={
        "has_image": "parse_image_path",
        "text_only": "extract_location_from_text"
    }
)

builder.add_edge("parse_image_path", "caption_image")
builder.add_edge("caption_image", "extract_location_from_caption")
builder.add_edge("extract_location_from_caption", "search_weather")
builder.add_edge("extract_location_from_text", "search_weather")

builder.add_edge("search_weather", "summarize")
builder.add_edge("summarize", END)

graph = builder.compile()

In [ ]:
from IPython.display import Image as img, display

try:
    display(img(graph.get_graph().draw_mermaid_png()))
except Exception as e:
    # This requires some extra dependencies and is optional
    print(e)
    pass

## エージェントの実行

#### 画像を使用するユースケース

In [ ]:
# With image
result = graph.invoke({"input": "この写真の場所の天気は？ ./imgs/place_pic.jpg"})

# Print all steps
for k, v in result.items():
    print(f"{k.upper()}:\n{v}\n{'-'*50}")

#### 画像を使用しないユースケース

In [ ]:
# Without image
result = graph.invoke({"input": "今日の東京の天気を調べて"})

# Print all steps
for k, v in result.items():
    print(f"{k.upper()}:\n{v}\n{'-'*50}")

In [ ]:
from langchain_core.runnables import RunnableConfig

from pprint import pprint

inputs = {
    "input": "この写真の場所の天気は？ ./imgs/place_pic1.jpg"
}

from langchain_core.callbacks.base import BaseCallbackHandler

class PrintStepCallback(BaseCallbackHandler):
    def on_tool_start(self, serialized, input_str, **kwargs):
        print(f"\n🔧 Tool Call: {serialized['name']} with input → {input_str}")

    def on_tool_end(self, output, **kwargs):
        print(f"✅ Tool Output: {output}")

    def on_chain_end(self, outputs, **kwargs):
        print(f"\n🔚 Final Chain Output: {outputs}")

config = RunnableConfig(callbacks=[PrintStepCallback()])
state = graph.invoke(inputs, config=config)

print("\n✅ Final Result:")
print(state['final_response'])

In [ ]:
! docker stop llm_nim

In [ ]:
! docker stop vlm_nim

## 演習

エージェントフローにカスタム操作を実行する独自のツールを作成して追加してください。
例えば:
* 計算ツールを作成する (LLMは数学的計算が苦手なため、長い数式の評価に使用できます)
* 場所を検索して地図上に配置できるジオロケーターツールを作成する
* Stable Diffusionなどのtext2imgモデルを使用して、その場所の有名なランドマークのアニメーションレプリカを作成するツールを作成する

このノートブックでは、NVIDIA NIMが複数の異なるツールや他のNVIDIA NIMと連携して、エージェントワークフローを作成できることを示しました。

---
## Licensing

Copyright © 2025 OpenACC-Standard.org. This material is released by OpenACC-Standard.org, in collaboration with NVIDIA Corporation, under the Creative Commons Attribution 4.0 International (CC BY 4.0). These materials include references to hardware and software developed by other entities; all applicable licensing and copyrights apply.